In [ ]:
import matplotlib.pyplot as plt
from itertools import combinations

from test.test_radial_collimator import make_test_instrument
from test.utilities import compile_and_scan

In [ ]:
instr = make_test_instrument()

In [ ]:
results = {}
# collimators = ("Collimator_Radial", "Collimator_ROC", "Exact_radial_coll")
# acceptable = {k: v for k, v in zip(collimators, (1.52649e-08, 1.41509e-08,1.581e-08))}

collimators = {
        "Collimator_Radial": 1.52649e-08, 
        "Collimator_ROC": 1.41509e-08, 
        "Exact_radial_coll": 1.581e-08,
        "Radial_filter_collimator": 1.42e-08,  # should match Exact_radial_coll since it's configured to be the same geometry with vacuum
        }

res_dict = compile_and_scan(instr, dict(collimator="1,2,3,4"), ncount=1e7, seed=1)

In [ ]:
res_dict
colnames = list(collimators.keys())
banana_theta = {colnames[i]: v['banana_theta'] for i, v in enumerate(res_dict['scan_result'])}

totals = {k: sum(v['I']) for k, v in banana_theta.items()}

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from itertools import combinations

In [ ]:
def is_acceptable(collimator):
    return abs(totals[collimator] / collimators[collimator] - 1) < 0.04

In [ ]:
fig, ax = plt.subplots(len(collimators), len(collimators), sharex=True, sharey=True)
fig.set_size_inches(13*4/3, 8*4/3)
for i, (c, v) in enumerate(banana_theta.items()):
    ax[0,i].plot(v['A'], v['I'], label=c)
    ax[0,i].legend()
    plt.setp(ax[0,i], facecolor=(0,1,0,0.1) if is_acceptable(c) else (1,0,0,0.1))

for (i, f), (j, s) in combinations(enumerate(collimators), 2):
    x = banana_theta[f]['A']
    y = banana_theta[f]['I'] - banana_theta[s]['I']
    t = totals[f]
    ax[j, i].plot(x, y, label=f'{f}-{s}')
    ax[j, i].text(10, 2e-11, rf'$\Delta$ = {100 * abs(sum(y)/t): 4.1f}%')
    ax[j, i].legend()
